[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/InfiniteTalk_Colab.ipynb)

# 🎬 InfiniteTalk — Sparse-Frame Audio-Driven Video Generation & Dubbing (Colab Edition)

**Model:** [`MeiGen-AI/InfiniteTalk`](https://huggingface.co/MeiGen-AI/InfiniteTalk) &nbsp;|&nbsp; **Base Backbone:** [`Wan-AI/Wan2.1-I2V-14B-480P`](https://huggingface.co/Wan-AI/Wan2.1-I2V-14B-480P) &nbsp;|&nbsp; **GitHub:** [`MeiGen-AI/InfiniteTalk`](https://github.com/MeiGen-AI/InfiniteTalk)

**InfiniteTalk** is MeiGen-AI's sparse-frame audio-driven video generation framework built on **Wan2.1-I2V-14B**. Unlike traditional talking-head or mouth-inpainting dubbing models, InfiniteTalk synchronizes **lip movements, facial expressions, head pose, and upper-body posture** across unlimited-length videos (`streaming` mode) at **480p** and **720p**.

### ✨ Key Features of This Colab Notebook
1. **3 Workflows Out of the Box**:
   - **🖼️ Single-Person Image-to-Video (`I2V`)**: Animate any portrait or character image with speech or singing audio of arbitrary length.
   - **🎞️ Sparse-Frame Video Dubbing (`V2V`)**: Lip-, head-, and body-sync an existing video (`.mp4`) to new audio while preserving camera motion and identity (with optional `PySceneDetect` shot splitting and Lab color-drift correction).
   - **👥 Multi-Speaker Conversation (`Multi-I2V` / `Multi-V2V`)**: Drive two speakers in the same scene using two separate audio tracks in either **sequential turn-taking (`add`)** or **parallel (`para`)** mode with automatic left/right or custom bounding-box routing.
2. **⚡ Fast 8-Step FusionX LoRA Default (`7.5x` Faster Inference)**:
   - Uses `MeiGen-AI/InfiniteTalk`'s official pre-merged **INT8/FP8 + FusionX LoRA** checkpoints (`sampling_steps=8`, `shift=2.0`, `text_guide_scale=1.0`, `audio_guide_scale=2.0`), requiring only **16 DiT forward passes per chunk instead of 120**.
   - Switchable via a single dropdown to the **40-Step Standard** checkpoint for maximum identity lock and >60s color stability.
3. **💾 Smart Selective Downloader (`~30.5 GB` vs `~258 GB`)**:
   - Downloads only the pre-quantized DiT (`18.6 GB`), quantized `t5_fp8` (`6.4 GB`), Wan2.1 VAE + CLIP + tokenizers (`5.1 GB`), and `chinese-wav2vec2-base` (`0.36 GB`)—skipping `~228 GB` of redundant unquantized shards.
4. **🛠️ 7 Built-In Upstream Fixes for Single-GPU Colab (L4 24GB & A100 40GB/80GB)**:
   - Automatic PyTorch 2.x SDPA fallback (no `flash_attn` or `xfuser` compilation required), `quant="int8"` T5 fallback to `t5_fp8`, removal of 240 redundant `torch.cuda.empty_cache()` calls per step, accurate `optimum-quanto` VRAM budgeting on 24 GB GPUs, automatic short-audio (`<=3.2s`) padding/trimming, and 2-pass TeaCache sync.

---
## Step 1 — Mount Google Drive, Check GPU Hardware & Configure Paths
Run this cell first to verify your Colab GPU (L4 24GB or A100 40GB/80GB recommended) and configure persistent directories on Google Drive so weights (`~30.5 GB`) only need to be downloaded once.

In [ ]:
#@title 1. Mount Google Drive & Inspect Hardware { display-mode: "form" }
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_BASE_DIR = "/content/drive/MyDrive/InfiniteTalk_Colab" #@param {type:"string"}
LOCAL_REPO_DIR = "/content/InfiniteTalk" #@param {type:"string"}

import os
import sys
import shutil
import psutil
import torch

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        WORKSPACE_DIR = DRIVE_BASE_DIR
        print(f"✅ Google Drive mounted. Workspace: {WORKSPACE_DIR}")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive ({e}). Falling back to /content/InfiniteTalk_Colab")
        WORKSPACE_DIR = "/content/InfiniteTalk_Colab"
else:
    WORKSPACE_DIR = "/content/InfiniteTalk_Colab"
    print(f"ℹ️ Using local Colab storage: {WORKSPACE_DIR}")

WEIGHTS_DIR = os.path.join(WORKSPACE_DIR, "weights")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")
INPUT_DIR = os.path.join(WORKSPACE_DIR, "inputs")
HF_CACHE_DIR = os.path.join(WORKSPACE_DIR, "hf_cache")
AUDIO_TEMP_DIR = "/content/infinitetalk_audio_tmp"

for d in [WORKSPACE_DIR, WEIGHTS_DIR, OUTPUT_DIR, INPUT_DIR, HF_CACHE_DIR, AUDIO_TEMP_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if not torch.cuda.is_available():
    raise RuntimeError("❌ No CUDA GPU detected! In Colab, go to Runtime -> Change runtime type -> select L4 GPU or A100 GPU.")

gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
ram_total_gb = psutil.virtual_memory().total / (1024 ** 3)
disk_free_gb = shutil.disk_usage(WORKSPACE_DIR).free / (1024 ** 3)

print("=" * 68)
print(f"🖥️  GPU Device         : {gpu_name} (Compute Capability {cc_major}.{cc_minor})")
print(f"🧠  GPU VRAM           : {vram_total_gb:.2f} GB")
print(f"📦  System Host RAM    : {ram_total_gb:.2f} GB")
print(f"💾  Free Disk Space    : {disk_free_gb:.2f} GB (at {WORKSPACE_DIR})")
print(f"🔥  PyTorch / CUDA     : {torch.__version__} / CUDA {torch.version.cuda}")
print("=" * 68)

if vram_total_gb >= 38.0:
    RECOMMENDED_PERSISTENT_PARAMS = None  # Full quantized DiT stays on GPU
    print("🚀 A100-class VRAM detected (>=38 GB): Full quantized DiT can stay resident in GPU VRAM!")
elif vram_total_gb >= 21.0:
    RECOMMENDED_PERSISTENT_PARAMS = 6_000_000_000  # ~6.5 GB persistent on GPU, rest streamed via AutoWrappedQLinear
    print("⚡ L4/24GB-class VRAM detected: Will pin ~6B DiT parameters in VRAM and stream overflow layers cleanly.")
else:
    RECOMMENDED_PERSISTENT_PARAMS = 0  # Full layer-by-layer CPU->GPU streaming
    print("⚠️ <21 GB VRAM detected: Will use full layer-by-layer DiT offloading (num_persistent_param_in_dit = 0).")

---
## Step 2 — Clone Repository, Install Dependencies & Apply 7 Colab Patches
This cell clones [`MeiGen-AI/InfiniteTalk`](https://github.com/MeiGen-AI/InfiniteTalk), installs the required Python packages without disturbing Colab's pre-installed PyTorch/CUDA build, and automatically applies **7 patches** for single-GPU Colab execution:
1. **`wan/modules/t5.py`**: Falls back to `t5_fp8.safetensors` + `t5_map_fp8.json` when `quant="int8"` is selected (since `MeiGen-AI/InfiniteTalk/quant_models/` only ships `t5_fp8`).
2. **`wan/modules/attention.py`**: Makes `xfuser` and `xformers` optional and adds a full PyTorch 2.x `F.scaled_dot_product_attention` fallback inside `flash_attention(...)` and `SingleStreamAttention`.
3. **`wan/utils/multitalk_utils.py` & `wan/modules/multitalk_model.py`**:
   - Skips computing unused reference attention maps (`get_attn_map_with_target`) across all 40 blocks when `human_num == 1`, and removes the inner-loop `torch_gc()` (`torch.cuda.empty_cache()`) inside `@torch.compile def calculate_x_ref_attn_map` (eliminating **240 CUDA synchronizations per forward pass**).
   - Updates TeaCache step tracking to support both 2-pass CFG (`text_guide_scale == 1.0` in 8-step LoRA mode) and 3-pass CFG.
4. **`src/vram_management/layers.py`**: Counts `optimum-quanto` `QLinear` quantized weight elements (`module.weight.numel()`) so `num_persistent_param_in_dit` accurately budgets VRAM on 24 GB L4 GPUs.
5. **`wan/multitalk.py`**: Adds safe `diffusers >= 0.33+` imports, sets `self.vram_management = True` inside `enable_vram_management()`, and allows exact-length audio embeddings (`< frame_num` instead of `<= frame_num`).
6. **`src/audio_analysis/wav2vec2.py`**: Disables unused `self.config.output_attentions = True` for compatibility with modern `transformers` SDPA.
7. **`wan/utils/segvideo.py`**: Replaces `moviepy.editor` subclipping with fast `ffmpeg` slicing for `scene_seg` compatibility across `moviepy` 1.x and 2.x.

In [ ]:
#@title 2. Clone InfiniteTalk, Install Packages & Apply Patches { display-mode: "form" }
import os
import sys
import subprocess

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists(os.path.join(LOCAL_REPO_DIR, "wan", "multitalk.py")):
    print(f"📥 Cloning MeiGen-AI/InfiniteTalk into {LOCAL_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/MeiGen-AI/InfiniteTalk.git", LOCAL_REPO_DIR],
        check=True,
    )
else:
    print(f"✅ Repository already present at {LOCAL_REPO_DIR}")

if LOCAL_REPO_DIR not in sys.path:
    sys.path.insert(0, LOCAL_REPO_DIR)

print("📦 Installing dependencies (preserving Colab PyTorch & CUDA)...")
pip_pkgs = [
    "optimum-quanto==0.2.6",
    "pyloudnorm",
    "scenedetect[opencv]",
    "decord",
    "librosa",
    "soundfile",
    "einops",
    "easydict",
    "ftfy",
    "scikit-image",
    "loguru",
    "imageio",
    "imageio-ffmpeg",
    "huggingface_hub[hf_transfer]",
    "diffusers>=0.31.0",
    "transformers>=4.49.0",
    "accelerate>=1.1.1",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pip_pkgs, check=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ------------------------------------------------------------------------------
# Patch 1: wan/modules/t5.py — fallback to t5_fp8 when quant="int8"
# ------------------------------------------------------------------------------
t5_path = os.path.join(LOCAL_REPO_DIR, "wan", "modules", "t5.py")
with open(t5_path, "r", encoding="utf-8") as f:
    t5_code = f.read()

old_t5_block = '''            logging.info(f'Loading quantized T5 from {os.path.join(quant_dir, f"t5_{quant}.safetensors")}')
            model_state_dict = load_file(os.path.join(quant_dir, f"t5_{quant}.safetensors"))
            with open(os.path.join(quant_dir, f"t5_map_{quant}.json"), "r") as f:
                quantization_map = json.load(f)'''

new_t5_block = '''            t5_Quant = quant if os.path.exists(os.path.join(quant_dir, f"t5_{quant}.safetensors")) else "fp8"
            logging.info(f'Loading quantized T5 from {os.path.join(quant_dir, f"t5_{t5_Quant}.safetensors")}')
            model_state_dict = load_file(os.path.join(quant_dir, f"t5_{t5_Quant}.safetensors"))
            with open(os.path.join(quant_dir, f"t5_map_{t5_Quant}.json"), "r") as f:
                quantization_map = json.load(f)'''

if old_t5_block in t5_code:
    t5_code = t5_code.replace(old_t5_block, new_t5_block)

old_t5_req = "            requantize(model, model_state_dict, quantization_map, device='cpu')"
new_t5_req = "            requantize(model, model_state_dict, quantization_map, device='cpu')\n            del model_state_dict"
if old_t5_req in t5_code and "del model_state_dict" not in t5_code:
    t5_code = t5_code.replace(old_t5_req, new_t5_req)

with open(t5_path, "w", encoding="utf-8") as f:
    f.write(t5_code)
print("✅ Patch 1 applied: wan/modules/t5.py (t5_fp8 automatic fallback + host RAM cleanup)")

# ------------------------------------------------------------------------------
# Patch 2: wan/modules/attention.py — optional xfuser/xformers + SDPA in flash_attention
# ------------------------------------------------------------------------------
attn_path = os.path.join(LOCAL_REPO_DIR, "wan", "modules", "attention.py")
with open(attn_path, "r", encoding="utf-8") as f:
    attn_code = f.read()

old_attn_imports = '''from xfuser.core.distributed import (
    get_sequence_parallel_rank,
    get_sequence_parallel_world_size,
    get_sp_group,
)
import xformers.ops'''

new_attn_imports = '''try:
    from xfuser.core.distributed import (
        get_sequence_parallel_rank,
        get_sequence_parallel_world_size,
        get_sp_group,
    )
except ImportError:
    def get_sequence_parallel_rank(): return 0
    def get_sequence_parallel_world_size(): return 1
    def get_sp_group(): return None

try:
    import xformers.ops
    XFORMERS_AVAILABLE = True
except ImportError:
    xformers = None
    XFORMERS_AVAILABLE = False'''

if old_attn_imports in attn_code:
    attn_code = attn_code.replace(old_attn_imports, new_attn_imports)

old_fa_else = '''    else:
        assert FLASH_ATTN_2_AVAILABLE
        x = flash_attn.flash_attn_varlen_func(
            q=q,
            k=k,
            v=v,
            cu_seqlens_q=torch.cat([q_lens.new_zeros([1]), q_lens]).cumsum(
                0, dtype=torch.int32).to(q.device, non_blocking=True),
            cu_seqlens_k=torch.cat([k_lens.new_zeros([1]), k_lens]).cumsum(
                0, dtype=torch.int32).to(q.device, non_blocking=True),
            max_seqlen_q=lq,
            max_seqlen_k=lk,
            dropout_p=dropout_p,
            softmax_scale=softmax_scale,
            causal=causal,
            window_size=window_size,
            deterministic=deterministic).unflatten(0, (b, lq))'''

new_fa_else = '''    elif FLASH_ATTN_2_AVAILABLE:
        x = flash_attn.flash_attn_varlen_func(
            q=q,
            k=k,
            v=v,
            cu_seqlens_q=torch.cat([q_lens.new_zeros([1]), q_lens]).cumsum(
                0, dtype=torch.int32).to(q.device, non_blocking=True),
            cu_seqlens_k=torch.cat([k_lens.new_zeros([1]), k_lens]).cumsum(
                0, dtype=torch.int32).to(q.device, non_blocking=True),
            max_seqlen_q=lq,
            max_seqlen_k=lk,
            dropout_p=dropout_p,
            softmax_scale=softmax_scale,
            causal=causal,
            window_size=window_size,
            deterministic=deterministic).unflatten(0, (b, lq))
    else:
        # Native PyTorch 2.x Flash/Memory-Efficient SDPA fallback
        if (q_lens is not None and not torch.all(q_lens == lq)) or (k_lens is not None and not torch.all(k_lens == lk)):
            out_list = []
            q_offset, k_offset = 0, 0
            for i in range(b):
                ql_i = int(q_lens[i].item())
                kl_i = int(k_lens[i].item())
                qi = q[q_offset:q_offset + ql_i].unsqueeze(0).transpose(1, 2)
                ki = k[k_offset:k_offset + kl_i].unsqueeze(0).transpose(1, 2)
                vi = v[k_offset:k_offset + kl_i].unsqueeze(0).transpose(1, 2)
                oi = torch.nn.functional.scaled_dot_product_attention(
                    qi, ki, vi, is_causal=causal, dropout_p=dropout_p, scale=softmax_scale
                ).transpose(1, 2).squeeze(0)
                if ql_i < lq:
                    oi = torch.cat([oi, oi.new_zeros(lq - ql_i, *oi.shape[1:])], dim=0)
                out_list.append(oi)
                q_offset += ql_i
                k_offset += kl_i
            x = torch.stack(out_list, dim=0)
        else:
            q_b = q.unflatten(0, (b, lq)).transpose(1, 2)
            k_b = k.unflatten(0, (b, lk)).transpose(1, 2)
            v_b = v.unflatten(0, (b, lk)).transpose(1, 2)
            x = torch.nn.functional.scaled_dot_product_attention(
                q_b, k_b, v_b, is_causal=causal, dropout_p=dropout_p, scale=softmax_scale
            ).transpose(1, 2).contiguous()'''

if old_fa_else in attn_code:
    attn_code = attn_code.replace(old_fa_else, new_fa_else)

old_xformers_call_1 = "x = xformers.ops.memory_efficient_attention(q, encoder_k, encoder_v, attn_bias=attn_bias, op=None,)"
new_xformers_call_1 = "x = xformers.ops.memory_efficient_attention(q, encoder_k, encoder_v, attn_bias=attn_bias, op=None) if (XFORMERS_AVAILABLE and attn_bias is not None) else torch.nn.functional.scaled_dot_product_attention(q.transpose(1, 2), encoder_k.transpose(1, 2), encoder_v.transpose(1, 2)).transpose(1, 2).contiguous()"
if old_xformers_call_1 in attn_code:
    attn_code = attn_code.replace(old_xformers_call_1, new_xformers_call_1)

old_xformers_call_2 = "x = xformers.ops.memory_efficient_attention(q, encoder_k, encoder_v, attn_bias=None, op=None,)"
new_xformers_call_2 = "x = torch.nn.functional.scaled_dot_product_attention(q.transpose(1, 2), encoder_k.transpose(1, 2), encoder_v.transpose(1, 2)).transpose(1, 2).contiguous()"
if old_xformers_call_2 in attn_code:
    attn_code = attn_code.replace(old_xformers_call_2, new_xformers_call_2)

with open(attn_path, "w", encoding="utf-8") as f:
    f.write(attn_code)
print("✅ Patch 2 applied: wan/modules/attention.py (PyTorch SDPA fallback + optional xfuser/xformers)")

# ------------------------------------------------------------------------------
# Patch 3: wan/utils/multitalk_utils.py & wan/modules/multitalk_model.py
#   - Optional xfuser import
#   - Remove inner-loop torch_gc() inside @torch.compile calculate_x_ref_attn_map
#   - Skip get_attn_map_with_target when ref_target_masks is None or human_num == 1
#   - Fix TeaCache 2-pass vs 3-pass CFG modulo & attribute initialization
# ------------------------------------------------------------------------------
mt_utils_path = os.path.join(LOCAL_REPO_DIR, "wan", "utils", "multitalk_utils.py")
with open(mt_utils_path, "r", encoding="utf-8") as f:
    mt_utils_code = f.read()

old_mtu_imports = '''from xfuser.core.distributed import (
    get_sequence_parallel_rank,
    get_sequence_parallel_world_size,
    get_sp_group,
)'''
new_mtu_imports = '''try:
    from xfuser.core.distributed import (
        get_sequence_parallel_rank,
        get_sequence_parallel_world_size,
        get_sp_group,
    )
except ImportError:
    def get_sequence_parallel_rank(): return 0
    def get_sequence_parallel_world_size(): return 1
    def get_sp_group(): return None'''

if old_mtu_imports in mt_utils_code:
    mt_utils_code = mt_utils_code.replace(old_mtu_imports, new_mtu_imports)

# Remove inner-loop torch_gc() inside @torch.compile calculate_x_ref_attn_map
old_calc_loop = '''    for class_idx, ref_target_mask in enumerate(ref_target_masks):
        torch_gc()
        ref_target_mask = ref_target_mask[None, None, None, ...]'''
new_calc_loop = '''    for class_idx, ref_target_mask in enumerate(ref_target_masks):
        ref_target_mask = ref_target_mask[None, None, None, ...]'''
if old_calc_loop in mt_utils_code:
    mt_utils_code = mt_utils_code.replace(old_calc_loop, new_calc_loop)

old_calc_end = '''    del attn
    del x_ref_attn_map_source
    torch_gc()

    return torch.concat(x_ref_attn_maps, dim=0)'''
new_calc_end = '''    del attn
    del x_ref_attn_map_source

    return torch.concat(x_ref_attn_maps, dim=0)'''
if old_calc_end in mt_utils_code:
    mt_utils_code = mt_utils_code.replace(old_calc_end, new_calc_end)

old_get_attn_start = '''def get_attn_map_with_target(visual_q, ref_k, shape, ref_target_masks=None, split_num=2, enable_sp=False):'''
new_get_attn_start = '''def get_attn_map_with_target(visual_q, ref_k, shape, ref_target_masks=None, split_num=2, enable_sp=False):
    if ref_target_masks is None:
        return None'''
if "if ref_target_masks is None:" not in mt_utils_code and old_get_attn_start in mt_utils_code:
    mt_utils_code = mt_utils_code.replace(old_get_attn_start, new_get_attn_start)

# Remove @torch.compile on calculate_x_ref_attn_map and @lru_cache(maxsize=32) on precompute_freqs_cis_1d (fixes 21.4 GB Multi-Speaker VRAM leak & tensor equality crash)
mt_utils_code = mt_utils_code.replace("@torch.compile\ndef calculate_x_ref_attn_map", "def calculate_x_ref_attn_map")
mt_utils_code = mt_utils_code.replace("    @lru_cache(maxsize=32)\n    def precompute_freqs_cis_1d", "    def precompute_freqs_cis_1d")

with open(mt_utils_path, "w", encoding="utf-8") as f:
    f.write(mt_utils_code)

mt_model_path = os.path.join(LOCAL_REPO_DIR, "wan", "modules", "multitalk_model.py")
with open(mt_model_path, "r", encoding="utf-8") as f:
    mt_model_code = f.read()

old_mask_conv = '''        # convert ref_target_masks to token_ref_target_masks
        if ref_target_masks is not None:'''
new_mask_conv = '''        # convert ref_target_masks to token_ref_target_masks (only needed when human_num > 1)
        token_ref_target_masks = None
        if ref_target_masks is not None and human_num is not None and human_num > 1:'''
if old_mask_conv in mt_model_code:
    mt_model_code = mt_model_code.replace(old_mask_conv, new_mask_conv)

old_teacache_init = '''        self.__class__.cnt = 0
        self.__class__.num_steps = sample_steps*3
        self.__class__.teacache_thresh = teacache_thresh
        self.__class__.accumulated_rel_l1_distance_even = 0
        self.__class__.accumulated_rel_l1_distance_odd = 0
        self.__class__.previous_e0_even = None
        self.__class__.previous_e0_odd = None
        self.__class__.previous_residual_even = None
        self.__class__.previous_residual_odd = None
        self.__class__.use_ret_steps = use_ret_steps'''

new_teacache_init = '''        passes = getattr(self.__class__, "passes_per_step", 3)
        self.__class__.cnt = 0
        self.__class__.num_steps = sample_steps * passes
        self.__class__.teacache_thresh = teacache_thresh
        self.__class__.accumulated_rel_l1_distance_cond = 0
        self.__class__.accumulated_rel_l1_distance_drop_text = 0
        self.__class__.accumulated_rel_l1_distance_uncond = 0
        self.__class__.previous_e0_cond = None
        self.__class__.previous_e0_drop_text = None
        self.__class__.previous_e0_uncond = None
        self.__class__.previous_residual_cond = None
        self.__class__.previous_residual_drop_text = None
        self.__class__.previous_residual_uncond = None
        self.__class__.use_ret_steps = use_ret_steps'''

if old_teacache_init in mt_model_code:
    mt_model_code = mt_model_code.replace(old_teacache_init, new_teacache_init)
    mt_model_code = mt_model_code.replace("self.__class__.ret_steps = 5*3", "self.__class__.ret_steps = 5 * getattr(self.__class__, 'passes_per_step', 3)")
    mt_model_code = mt_model_code.replace("self.__class__.cutoff_steps = sample_steps*3 - 3", "self.__class__.cutoff_steps = sample_steps * getattr(self.__class__, 'passes_per_step', 3) - getattr(self.__class__, 'passes_per_step', 3)")
    mt_model_code = mt_model_code.replace("self.__class__.cutoff_steps = sample_steps*3", "self.__class__.cutoff_steps = sample_steps * getattr(self.__class__, 'passes_per_step', 3)")
    mt_model_code = mt_model_code.replace("self.__class__.ret_steps = 1*3", "self.__class__.ret_steps = 1 * getattr(self.__class__, 'passes_per_step', 3)")
    mt_model_code = mt_model_code.replace("if self.cnt%3==0:", "if self.cnt % getattr(self.__class__, 'passes_per_step', 3) == 0:")
    mt_model_code = mt_model_code.replace("elif self.cnt%3==1:", "elif self.cnt % getattr(self.__class__, 'passes_per_step', 3) == 1:")

with open(mt_model_path, "w", encoding="utf-8") as f:
    f.write(mt_model_code)
print("✅ Patch 3 applied: multitalk_utils.py & multitalk_model.py (removed 240x empty_cache stalls + TeaCache 2/3-pass sync)")

# ------------------------------------------------------------------------------
# Patch 4: src/vram_management/layers.py — count QLinear weight elements accurately
# ------------------------------------------------------------------------------
vram_layers_path = os.path.join(LOCAL_REPO_DIR, "src", "vram_management", "layers.py")
with open(vram_layers_path, "r", encoding="utf-8") as f:
    vram_layers_code = f.read()

old_num_param = "num_param = sum(p.numel() for p in module.parameters())"
new_num_param = '''num_param = sum(p.numel() for p in module.parameters()) + sum(b.numel() for b in module.buffers())
                if hasattr(module, "weight") and module.weight is not None and hasattr(module.weight, "numel"):
                    num_param = max(num_param, module.weight.numel())'''

if old_num_param in vram_layers_code and "module. buffers()" not in vram_layers_code and "module.buffers()" not in vram_layers_code:
    vram_layers_code = vram_layers_code.replace(old_num_param, new_num_param)
    with open(vram_layers_path, "w", encoding="utf-8") as f:
        f.write(vram_layers_code)
print("✅ Patch 4 applied: src/vram_management/layers.py (accurate QLinear parameter counting for L4 VRAM pinning)")

# ------------------------------------------------------------------------------
# Patch 5: wan/multitalk.py — safe diffusers imports, self.vram_management=True, short audio fix
# ------------------------------------------------------------------------------
multitalk_path = os.path.join(LOCAL_REPO_DIR, "wan", "multitalk.py")
with open(multitalk_path, "r", encoding="utf-8") as f:
    multitalk_code = f.read()

old_diff_import = "from diffusers.models.modeling_utils import no_init_weights, ContextManagers"
new_diff_import = '''try:
    from diffusers.models.modeling_utils import no_init_weights, ContextManagers
except ImportError:
    from contextlib import ExitStack
    try:
        from transformers.modeling_utils import no_init_weights
    except ImportError:
        from contextlib import nullcontext as no_init_weights
    class ContextManagers:
        def __init__(self, managers): self.managers = managers
        def __enter__(self):
            self.stack = ExitStack()
            for m in self.managers: self.stack.enter_context(m)
            return self.stack
        def __exit__(self, *args): return self.stack.__exit__(*args)'''

if old_diff_import in multitalk_code and "class ContextManagers:" not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_diff_import, new_diff_import)

old_enable_vram = '''        self.enable_cpu_offload()'''
new_enable_vram = '''        self.vram_management = True
        self.enable_cpu_offload()'''
if new_enable_vram not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_enable_vram, new_enable_vram)

old_teacache_call = '''        if extra_args.use_teacache:
            self.model.teacache_init('''
new_teacache_call = '''        if extra_args.use_teacache:
            self.model.__class__.passes_per_step = 2 if math.isclose(text_guide_scale, 1.0) else 3
            self.model.teacache_init('''
if old_teacache_call in multitalk_code and "passes_per_step" not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_teacache_call, new_teacache_call)

old_audio_len_check = "if full_audio_emb.shape[0] <= frame_num:"
new_audio_len_check = "if full_audio_emb.shape[0] < frame_num:"
if old_audio_len_check in multitalk_code:
    multitalk_code = multitalk_code.replace(old_audio_len_check, new_audio_len_check)

old_mt_req = "            requantize(self.model, model_state_dict, quantization_map, device='cpu')"
new_mt_req = "            requantize(self.model, model_state_dict, quantization_map, device='cpu')\n            del model_state_dict\n            gc.collect()"
if old_mt_req in multitalk_code and "del model_state_dict" not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_mt_req, new_mt_req)

old_gen_list_init = "        gen_video_list = []"
new_gen_list_init = "        gen_video_list = []\n        miss_lengths = [0]"
if old_gen_list_init in multitalk_code and "miss_lengths = [0]" not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_gen_list_init, new_gen_list_init)

old_vae_offload = '''                if offload_model: 
                    if not self.vram_management:
                        self.model.cpu()'''
new_vae_offload = '''                if offload_model: 
                    if not self.vram_management:
                        self.model.cpu()
                    else:
                        self.load_models_to_device([])'''
if old_vae_offload in multitalk_code and "self.load_models_to_device([])" not in multitalk_code:
    multitalk_code = multitalk_code.replace(old_vae_offload, new_vae_offload)

with open(multitalk_path, "w", encoding="utf-8") as f:
    f.write(multitalk_code)
print("✅ Patch 5 applied: wan/multitalk.py (diffusers >=0.33 compatibility, self.vram_management=True, pre-VAE offload, audio frame check)")

# ------------------------------------------------------------------------------
# Patch 6: src/audio_analysis/wav2vec2.py — disable unused output_attentions=True
# ------------------------------------------------------------------------------
w2v_path = os.path.join(LOCAL_REPO_DIR, "src", "audio_analysis", "wav2vec2.py")
with open(w2v_path, "r", encoding="utf-8") as f:
    w2v_code = f.read()

if "self.config.output_attentions = True" in w2v_code:
    w2v_code = w2v_code.replace("self.config.output_attentions = True", "self.config.output_attentions = False")
    with open(w2v_path, "w", encoding="utf-8") as f:
        f.write(w2v_code)
print("✅ Patch 6 applied: src/audio_analysis/wav2vec2.py (compatibility with modern transformers SDPA)")

# ------------------------------------------------------------------------------
# Patch 7: wan/utils/segvideo.py — replace moviepy.editor with fast ffmpeg slicing
# ------------------------------------------------------------------------------
seg_path = os.path.join(LOCAL_REPO_DIR, "wan", "utils", "segvideo.py")
new_seg_code = '''from scenedetect import SceneManager, open_video, ContentDetector, AdaptiveDetector, ThresholdDetector
import os, time, subprocess

def build_manager():
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector())
    scene_manager.add_detector(AdaptiveDetector())
    scene_manager.add_detector(ThresholdDetector())
    return scene_manager

def seg_video(video_path, scene_list, output_dir):
    output_fp_list = []
    for (start_time, end_time) in scene_list:
        if end_time - start_time > 0.5:
            s_t = max(0.0, start_time + 0.05)
            e_t = end_time - 0.05
            vid = os.path.splitext(os.path.basename(video_path))[0].split('___')[0]
            output_fp = os.path.join(output_dir, f'{vid}_{s_t:.2f}_{e_t:.2f}.mp4')
            cmd = [
                'ffmpeg', '-y', '-ss', f'{s_t:.3f}', '-to', f'{e_t:.3f}',
                '-i', video_path, '-c:v', 'libx264', '-preset', 'fast', '-c:a', 'aac', output_fp
            ]
            subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            output_fp_list.append(output_fp)
    return output_fp_list

def shot_detect(video_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f'start process {video_path}')
    start_time = time.time()
    scene_list, output_fp_list = [], []
    try:
        video = open_video(video_path)
        scene_manager = build_manager()
        scene_manager.detect_scenes(video, show_progress=False)
        stamps = scene_manager.get_scene_list()
        for stamp in stamps:
            start, end = stamp
            scene_list.append((start.get_seconds(), end.get_seconds()))
        output_fp_list = seg_video(video_path, scene_list, output_dir)
    except Exception as e:
        print(f'shot_detect warning: {e} ({video_path})')
    print(f"process {video_path} Done with {time.time()-start_time:.2f} seconds used.")
    return scene_list, output_fp_list
'''
with open(seg_path, "w", encoding="utf-8") as f:
    f.write(new_seg_code)
print("✅ Patch 7 applied: wan/utils/segvideo.py (fast ffmpeg shot segmentation without moviepy.editor)")

---
## Step 3 — Smart Selective Model Downloader (`~30.5 GB` instead of `~258 GB`)
Instead of downloading all `~258 GB` of raw and quantized checkpoints across the three Hugging Face repositories, this cell selectively downloads **only** the exact files needed for your chosen speaker mode and quality preset:
- **`Wan-AI/Wan2.1-I2V-14B-480P` (`5.08 GB`)**: `config.json`, `Wan2.1_VAE.pth`, `models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth`, `google/umt5-xxl/*`, and `xlm-roberta-large/*` *(skips 73.4 GB of unquantized DiT/T5 weights)*.
- **`TencentGameMate/chinese-wav2vec2-base` (`0.36 GB`)**: `config.json`, `preprocessor_config.json`, and `pytorch_model.bin` *(skips 1.09 GB fairseq checkpoint)*.
- **`MeiGen-AI/InfiniteTalk` (`25.02 GB`)**:
  - Quantized UMT5-XXL text encoder: `quant_models/t5_fp8.safetensors` + `t5_map_fp8.json` (`6.42 GB`)
  - Selected pre-merged quantized InfiniteTalk DiT (`18.60 GB`):
    - **`fast_8step_lora` (Recommended)**: Pre-merged with the **FusionX 8-step distillation LoRA** (`7.5x` faster inference: 8 steps × 2 CFG passes = 16 DiT evaluations per chunk).
    - **`standard_40step`**: Standard 40-step checkpoint (best for videos >60 seconds or maximum identity preservation).

In [ ]:
#@title 3. Select Preset & Download Required Weights (~30.5 GB) { display-mode: "form" }
SPEAKER_MODE = "single" #@param ["single", "multi"]
QUALITY_PRESET = "fast_8step_lora" #@param ["fast_8step_lora", "standard_40step"]
QUANT_PRECISION = "int8" #@param ["int8", "fp8"]

import os
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download, hf_hub_download

WAN_CKPT_DIR = os.path.join(WEIGHTS_DIR, "Wan2.1-I2V-14B-480P")
WAV2VEC_DIR = os.path.join(WEIGHTS_DIR, "chinese-wav2vec2-base")
INFINITETALK_DIR = os.path.join(WEIGHTS_DIR, "InfiniteTalk")

os.makedirs(WAN_CKPT_DIR, exist_ok=True)
os.makedirs(WAV2VEC_DIR, exist_ok=True)
os.makedirs(INFINITETALK_DIR, exist_ok=True)

# 1. Wan2.1-I2V-14B-480P (Only VAE, CLIP, config, and tokenizers = 5.08 GB)
print("📥 [1/3] Checking Wan-AI/Wan2.1-I2V-14B-480P support files (~5.08 GB)...")
snapshot_download(
    repo_id="Wan-AI/Wan2.1-I2V-14B-480P",
    local_dir=WAN_CKPT_DIR,
    allow_patterns=[
        "config.json",
        "Wan2.1_VAE.pth",
        "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth",
        "google/umt5-xxl/*",
        "xlm-roberta-large/*",
    ],
)

# 2. TencentGameMate/chinese-wav2vec2-base (Only transformers weights = 0.36 GB)
print("📥 [2/3] Checking TencentGameMate/chinese-wav2vec2-base (~0.36 GB)...")
snapshot_download(
    repo_id="TencentGameMate/chinese-wav2vec2-base",
    local_dir=WAV2VEC_DIR,
    allow_patterns=[
        "config.json",
        "preprocessor_config.json",
        "pytorch_model.bin",
    ],
)

# 3. MeiGen-AI/InfiniteTalk (Quantized T5 + Selected Quantized DiT = 25.02 GB)
lora_suffix = "_lora" if QUALITY_PRESET == "fast_8step_lora" else ""
quant_stem = f"infinitetalk_{SPEAKER_MODE}_{QUANT_PRECISION}{lora_suffix}"
SELECTED_QUANT_MODEL_PATH = os.path.join(INFINITETALK_DIR, "quant_models", f"{quant_stem}.safetensors")
SELECTED_QUANT_JSON_PATH = os.path.join(INFINITETALK_DIR, "quant_models", f"{quant_stem}.json")

print(f"📥 [3/3] Checking MeiGen-AI/InfiniteTalk ({quant_stem} + t5_fp8, ~25.02 GB)...")
snapshot_download(
    repo_id="MeiGen-AI/InfiniteTalk",
    local_dir=INFINITETALK_DIR,
    allow_patterns=[
        "quant_models/t5_fp8.safetensors",
        "quant_models/t5_map_fp8.json",
        f"quant_models/{quant_stem}.safetensors",
        f"quant_models/{quant_stem}.json",
    ],
)

required_files = [
    os.path.join(WAN_CKPT_DIR, "config.json"),
    os.path.join(WAN_CKPT_DIR, "Wan2.1_VAE.pth"),
    os.path.join(WAN_CKPT_DIR, "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth"),
    os.path.join(WAV2VEC_DIR, "pytorch_model.bin"),
    os.path.join(INFINITETALK_DIR, "quant_models", "t5_fp8.safetensors"),
    SELECTED_QUANT_MODEL_PATH,
    SELECTED_QUANT_JSON_PATH,
]

print("\n📋 Verifying downloaded checkpoint files:")
for fp in required_files:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"❌ Missing required checkpoint file: {fp}")
    sz_mb = os.path.getsize(fp) / (1024 ** 2)
    print(f"  ✅ {os.path.relpath(fp, WEIGHTS_DIR):<62} ({sz_mb:>8.1f} MB)")

print(f"\n🎉 All weights ready! Active DiT checkpoint: {os.path.basename(SELECTED_QUANT_MODEL_PATH)}")

---
## Step 4 — Load `InfiniteTalkPipeline` & `Wav2Vec2` Audio Encoder
This cell initializes the `Wav2Vec2` audio encoder and `wan.InfiniteTalkPipeline` using `torch.device("meta")` + `optimum.quanto.requantize` so host RAM usage stays low (`~20 GB`).
- **`VRAM_BUDGET_MODE = "auto"`**:
  - **A100 (`>=38 GB` VRAM)**: Keeps the entire `18.6 GB` quantized DiT on GPU (`num_persistent_param_in_dit = None`) for maximum speed.
  - **L4 (`24 GB` VRAM)**: Pins `6,000,000,000` parameters (`~6.5 GB`) in GPU VRAM and streams the remaining layers on-the-fly via `AutoWrappedQLinear`, leaving `~15 GB` of VRAM headroom for attention activations and 3D VAE decoding.
  - **Low VRAM / 720p on 24GB**: Select `"low_vram_0_persistent"` to stream all DiT layers one-by-one (`num_persistent_param_in_dit = 0`).

In [ ]:
#@title 4. Initialize InfiniteTalkPipeline & Audio Encoder { display-mode: "form" }
VRAM_BUDGET_MODE = "auto" #@param ["auto", "full_gpu_resident (A100 40GB+)", "balanced_6B_persistent (L4 24GB 480p)", "low_vram_0_persistent (24GB 720p / T4)"]
T5_ON_CPU = False #@param {type:"boolean"}

import gc
import time
import torch
from transformers import Wav2Vec2FeatureExtractor
from src.audio_analysis.wav2vec2 import Wav2Vec2Model
import wan
from wan.configs import WAN_CONFIGS

# Free any existing pipeline before reloading
for var_name in ["wan_i2v", "audio_encoder", "wav2vec_feature_extractor"]:
    if var_name in globals() and globals()[var_name] is not None:
        print(f"♻️ Releasing previous `{var_name}` instance from memory...")
        del globals()[var_name]
gc.collect()
torch.cuda.empty_cache()

t0 = time.time()
print("🎙️ Loading chinese-wav2vec2-base audio feature extractor...")
wav2vec_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC_DIR, local_files_only=True)
audio_encoder = Wav2Vec2Model.from_pretrained(
    WAV2VEC_DIR,
    local_files_only=True,
    attn_implementation="eager",
).to("cpu").eval()
audio_encoder.feature_extractor._freeze_parameters()

print(f"🎬 Loading InfiniteTalkPipeline ({os.path.basename(SELECTED_QUANT_MODEL_PATH)})...")
cfg = WAN_CONFIGS["infinitetalk-14B"]

wan_i2v = wan.InfiniteTalkPipeline(
    config=cfg,
    checkpoint_dir=WAN_CKPT_DIR,
    quant_dir=SELECTED_QUANT_MODEL_PATH,
    device_id=0,
    rank=0,
    t5_fsdp=False,
    dit_fsdp=False,
    use_usp=False,
    t5_cpu=T5_ON_CPU,
    quant=QUANT_PRECISION,
    dit_path=None,
    infinitetalk_dir=None,
)

# Determine persistent parameter budget in DiT
if VRAM_BUDGET_MODE == "auto":
    persistent_params = RECOMMENDED_PERSISTENT_PARAMS
elif VRAM_BUDGET_MODE.startswith("full_gpu"):
    persistent_params = None
elif VRAM_BUDGET_MODE.startswith("balanced_6B"):
    persistent_params = 6_000_000_000
else:
    persistent_params = 0

if persistent_params is not None:
    print(f"⚙️ Enabling VRAM management (num_persistent_param_in_dit = {persistent_params:,})...")
    wan_i2v.vram_management = True
    wan_i2v.enable_vram_management(num_persistent_param_in_dit=persistent_params)
else:
    print("🚀 Keeping full quantized DiT resident on GPU (with T5/CLIP offloaded when idle)...")
    wan_i2v.vram_management = False

# Offload CLIP vision encoder to CPU while idle (generate_infinitetalk moves it to GPU on demand)
wan_i2v.clip.model.cpu()
torch.cuda.empty_cache()

LOADED_SPEAKER_MODE = SPEAKER_MODE
LOADED_QUALITY_PRESET = QUALITY_PRESET
LOADED_QUANT_PRECISION = QUANT_PRECISION
LOADED_PERSISTENT_PARAMS = persistent_params

vram_alloc_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
print(f"✅ Pipeline loaded in {time.time() - t0:.1f}s! Current GPU VRAM allocated: {vram_alloc_gb:.2f} GB")

---
## Step 5 — Prepare Input Media (Image, Video, or Multi-Speaker Audio)
Choose one of the built-in examples from the official repository, provide custom file paths on Google Drive, or upload your own files directly from your browser:
- **`demo_single_singing_image` (`I2V`)**: Studio singer portrait (`examples/single/ref_image.png`) + singing audio (`examples/single/1.wav`).
- **`demo_single_talking_video` (`V2V Dubbing`)**: Reference talking video (`examples/single/ref_video.mp4`) + new audio track (`examples/single/1.wav`).
- **`demo_multi_car_conversation` (`Multi-Speaker I2V`)**: Two-person car scene (`examples/multi/ref_img.png`) + separate male & female audio tracks (`1-man.WAV`, `1-woman.WAV`). *(Requires `SPEAKER_MODE = "multi"` in Steps 3–4).*
- **`custom_paths` / `upload_files`**: Use any `.png`/`.jpg` image or `.mp4`/`.mov` video along with `.wav`/`.mp3` audio files (or even pass a video file as the audio source to automatically extract its audio track!).

In [ ]:
#@title 5. Select or Upload Reference Image/Video & Audio { display-mode: "form" }
INPUT_SOURCE = "demo_single_singing_image" #@param ["demo_single_singing_image", "demo_single_talking_video", "demo_multi_car_conversation", "latest_drive_tts_audio", "custom_paths", "upload_files"]

#@markdown ### Custom File Paths (used when `INPUT_SOURCE = "custom_paths"` or `"latest_drive_tts_audio"`)
CUSTOM_COND_VIDEO_OR_IMAGE = "" #@param {type:"string"}
CUSTOM_AUDIO_PERSON1 = "" #@param {type:"string"}
CUSTOM_AUDIO_PERSON2 = "" #@param {type:"string"}

#@markdown ### Multi-Speaker Options (used when `CUSTOM_AUDIO_PERSON2` is provided or `demo_multi_car_conversation` is selected)
MULTI_AUDIO_TYPE = "add" #@param ["add", "para"]
#@markdown - `add`: Person 1 speaks first, followed sequentially by Person 2.
#@markdown - `para`: Person 1 and Person 2 audio tracks play in parallel (equalized to the same duration automatically).
CUSTOM_BBOX_PERSON1 = "" #@param {type:"string"}
CUSTOM_BBOX_PERSON2 = "" #@param {type:"string"}
#@markdown *(Optional bounding boxes as `x_min, y_min, x_max, y_max` in pixel coordinates, where `x` is vertical row `[0..H]` and `y` is horizontal col `[0..W]` per InfiniteTalk's convention. Leave blank for automatic left/right person split.)*

import os
import json
import librosa
import numpy as np
from PIL import Image, ImageDraw
import IPython.display as ipd
from wan.utils.utils import extract_specific_frames, is_video

def discover_aei_drive_audio(max_files=50):
    """Scan AEI TTS, Voice Conversion, Music, and Post-Processor Google Drive folders for recent audio."""
    search_roots = [
        INPUT_DIR,
        "/content/drive/MyDrive/AEI_TTS_Out",
        "/content/drive/MyDrive/AEI_3D_Out",
        "/content/drive/MyDrive/AEI_TTS_Cache/voices",
    ]
    aud_exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    found = []
    for root_dir in search_roots:
        if not os.path.isdir(root_dir):
            continue
        for dirpath, _, filenames in os.walk(root_dir):
            for fn in filenames:
                if os.path.splitext(fn)[1].lower() in aud_exts:
                    full_p = os.path.join(dirpath, fn)
                    try:
                        mtime = os.path.getmtime(full_p)
                        found.append((mtime, full_p))
                    except OSError:
                        pass
    found.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in found[:max_files]]

recent_drive_audios = discover_aei_drive_audio()
if recent_drive_audios:
    print(f"🔎 Found {len(recent_drive_audios)} audio file(s) across AEI Drive folders (latest: {recent_drive_audios[0]})")

DEFAULT_DEMO_PROMPTS = {
    "demo_single_singing_image": (
        "A woman is passionately singing into a professional microphone in a recording studio. "
        "She wears large black headphones and a dark cardigan over a gray top. Her long, wavy brown "
        "hair frames her face as she looks slightly upwards, her mouth open mid-song. The studio is "
        "equipped with various audio equipment, including a mixing console and a keyboard, with "
        "soundproofing panels on the walls. The lighting is warm and focused on her, creating a "
        "professional and intimate atmosphere. A close-up shot captures her expressive performance."
    ),
    "demo_single_talking_video": "A man is talking naturally with expressive facial movements.",
    "demo_multi_car_conversation": (
        "In a casual, intimate setting, a man and a woman are engaged in a heartfelt conversation "
        "inside a car. The man, sporting a denim jacket over a blue shirt, sits attentively with a "
        "seatbelt fastened, his gaze fixed on the woman beside him. The woman, wearing a black tank "
        "top and a denim jacket draped over her shoulders, smiles warmly, her eyes reflecting genuine "
        "interest and connection. A medium shot from a slightly angled perspective, focusing on their "
        "expressions and body language."
    ),
}

if INPUT_SOURCE == "demo_single_singing_image":
    cond_video_path = os.path.join(LOCAL_REPO_DIR, "examples", "single", "ref_image.png")
    audio1_path = os.path.join(LOCAL_REPO_DIR, "examples", "single", "1.wav")
    audio2_path = None
    default_prompt = DEFAULT_DEMO_PROMPTS["demo_single_singing_image"]
elif INPUT_SOURCE == "demo_single_talking_video":
    cond_video_path = os.path.join(LOCAL_REPO_DIR, "examples", "single", "ref_video.mp4")
    audio1_path = os.path.join(LOCAL_REPO_DIR, "examples", "single", "1.wav")
    audio2_path = None
    default_prompt = DEFAULT_DEMO_PROMPTS["demo_single_talking_video"]
elif INPUT_SOURCE == "demo_multi_car_conversation":
    cond_video_path = os.path.join(LOCAL_REPO_DIR, "examples", "multi", "ref_img.png")
    audio1_path = os.path.join(LOCAL_REPO_DIR, "examples", "multi", "1-man.WAV")
    audio2_path = os.path.join(LOCAL_REPO_DIR, "examples", "multi", "1-woman.WAV")
    default_prompt = DEFAULT_DEMO_PROMPTS["demo_multi_car_conversation"]
elif INPUT_SOURCE == "latest_drive_tts_audio":
    if not recent_drive_audios:
        raise FileNotFoundError("❌ No recent audio files found in AEI Drive folders (/content/drive/MyDrive/AEI_TTS_Out, AEI_3D_Out, etc.).")
    cond_video_path = CUSTOM_COND_VIDEO_OR_IMAGE.strip() or os.path.join(LOCAL_REPO_DIR, "examples", "single", "ref_image.png")
    audio1_path = recent_drive_audios[0]
    audio2_path = CUSTOM_AUDIO_PERSON2.strip() or None
    default_prompt = "A person is talking naturally with realistic facial expressions and head movements."
elif INPUT_SOURCE == "custom_paths":
    cond_video_path = CUSTOM_COND_VIDEO_OR_IMAGE.strip()
    audio1_path = recent_drive_audios[0] if CUSTOM_AUDIO_PERSON1.strip().upper() == "LATEST_DRIVE_AUDIO" and recent_drive_audios else CUSTOM_AUDIO_PERSON1.strip()
    audio2_path = (recent_drive_audios[1] if CUSTOM_AUDIO_PERSON2.strip().upper() == "LATEST_DRIVE_AUDIO" and len(recent_drive_audios) > 1 else CUSTOM_AUDIO_PERSON2.strip()) or None
    default_prompt = "A person is talking naturally with realistic facial expressions and head movements."
elif INPUT_SOURCE == "upload_files":
    from google.colab import files
    print("📤 Upload 1) Reference Image (.png/.jpg) or Video (.mp4) AND 2) Audio file(s) (.wav/.mp3):")
    uploaded = files.upload()
    img_vid_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".mp4", ".mov", ".avi", ".mkv", ".webm"}
    aud_exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    cond_video_path, audio_candidates = None, []
    for fname, data in uploaded.items():
        save_p = os.path.join(INPUT_DIR, fname)
        with open(save_p, "wb") as f:
            f.write(data)
        ext = os.path.splitext(fname)[1].lower()
        if ext in aud_exts:
            audio_candidates.append(save_p)
        elif ext in img_vid_exts and cond_video_path is None:
            cond_video_path = save_p
        else:
            audio_candidates.append(save_p)
    if not cond_video_path or not audio_candidates:
        raise ValueError("❌ Please upload at least one image/video file and one audio file.")
    audio1_path = audio_candidates[0]
    audio2_path = audio_candidates[1] if len(audio_candidates) > 1 else None
    default_prompt = "A person is talking naturally with realistic facial expressions and head movements."

if not os.path.exists(cond_video_path):
    raise FileNotFoundError(f"❌ Reference image/video not found: {cond_video_path}")
if not os.path.exists(audio1_path):
    raise FileNotFoundError(f"❌ Audio 1 file not found: {audio1_path}")
if audio2_path and not os.path.exists(audio2_path):
    raise FileNotFoundError(f"❌ Audio 2 file not found: {audio2_path}")

is_multi_input = audio2_path is not None
if is_multi_input and globals().get("LOADED_SPEAKER_MODE") == "single":
    print("⚠️ WARNING: Two audio tracks are selected (Multi-Speaker), but the loaded checkpoint is `single`!")
    print("   👉 For true 2-speaker spatial routing, set SPEAKER_MODE = 'multi' in Step 3 and re-run Steps 3 & 4.")
elif (not is_multi_input) and globals().get("LOADED_SPEAKER_MODE") == "multi":
    print("ℹ️ Note: Single audio track selected while `multi` checkpoint is loaded.")

def _parse_bbox(bbox_str):
    if not bbox_str or not bbox_str.strip():
        return None
    parts = [float(x.strip()) for x in bbox_str.split(",")]
    if len(parts) != 4:
        raise ValueError(f"Bounding box must have 4 comma-separated numbers (x_min, y_min, x_max, y_max), got: {bbox_str}")
    return parts

bbox_dict = None
if is_multi_input:
    b1 = _parse_bbox(CUSTOM_BBOX_PERSON1)
    b2 = _parse_bbox(CUSTOM_BBOX_PERSON2)
    if b1 and b2:
        bbox_dict = {"person1": b1, "person2": b2}

# Inspect first frame & audio duration
first_frame_pil = extract_specific_frames(cond_video_path, 0)
dur1 = librosa.get_duration(path=audio1_path)
dur2 = librosa.get_duration(path=audio2_path) if audio2_path else 0.0
if is_multi_input:
    total_audio_dur = (dur1 + dur2) if MULTI_AUDIO_TYPE == "add" else max(dur1, dur2)
else:
    total_audio_dur = dur1
est_frames = int(round(total_audio_dur * 25))

ACTIVE_INPUT = {
    "cond_video": cond_video_path,
    "audio1": audio1_path,
    "audio2": audio2_path,
    "audio_type": MULTI_AUDIO_TYPE if is_multi_input else None,
    "bbox": bbox_dict,
    "default_prompt": default_prompt,
    "is_video": is_video(cond_video_path),
    "total_audio_dur": total_audio_dur,
    "est_frames": est_frames,
}

print("=" * 68)
print(f"🎬 Condition Input : {cond_video_path} ({'Video V2V' if ACTIVE_INPUT['is_video'] else 'Image I2V'}, {first_frame_pil.width}x{first_frame_pil.height})")
print(f"🔊 Person 1 Audio  : {audio1_path} ({dur1:.2f}s)")
if is_multi_input:
    print(f"🔊 Person 2 Audio  : {audio2_path} ({dur2:.2f}s, mode='{MULTI_AUDIO_TYPE}')")
print(f"⏱️ Total Duration  : {total_audio_dur:.2f}s (~{est_frames} frames at 25 FPS)")
print("=" * 68)

# Render preview image (with bounding boxes if multi-person)
preview_img = first_frame_pil.copy()
if is_multi_input:
    draw = ImageDraw.Draw(preview_img)
    h, w = preview_img.height, preview_img.width
    if bbox_dict:
        x1, y1, x2, y2 = bbox_dict["person1"]
        draw.rectangle([y1, x1, y2, x2], outline="#00ffcc", width=4)
        x1, y1, x2, y2 = bbox_dict["person2"]
        draw.rectangle([y1, x1, y2, x2], outline="#ff6600", width=4)
    else:
        fs = 0.05
        xm0, xm1 = int(h * fs), int(h * (1 - fs))
        ly0, ly1 = int((w // 2) * fs), int((w // 2) * (1 - fs))
        ry0, ry1 = int((w // 2) * fs + (w // 2)), int((w // 2) * (1 - fs) + (w // 2))
        draw.rectangle([ly0, xm0, ly1, xm1], outline="#00ffcc", width=4)
        draw.rectangle([ry0, xm0, ry1, xm1], outline="#ff6600", width=4)

preview_img.thumbnail((480, 480))
display(preview_img)
print("🎵 Person 1 Audio Preview:")
display(ipd.Audio(audio1_path))
if is_multi_input:
    print("🎵 Person 2 Audio Preview:")
    display(ipd.Audio(audio2_path))

---
## Step 6 — Generate InfiniteTalk Video (`streaming` Unlimited Length or `clip` Preview)
Run this cell to generate your talking or dubbed video.
- **`GENERATION_MODE`**:
  - **`"streaming"` (Default)**: Generates the full video iteratively across overlapping 81-frame chunks (`motion_frame = 9`) up to `MAX_DURATION_SECONDS` (or the full audio duration if `MAX_DURATION_SECONDS = 0`).
  - **`"clip"`**: Generates a single 81-frame (`3.24s` at 25 FPS) chunk for fast prompt/voice testing.
- **Automatic Preset Hyperparameters (`SAMPLING_STEPS = 0`)**:
  - When **`fast_8step_lora`** is loaded: Automatically selects **`steps = 8`**, **`shift = 2.0`**, **`text_guide_scale = 1.0`**, **`audio_guide_scale = 2.0`** (`7.5x` faster!).
  - When **`standard_40step`** is loaded: Automatically selects **`steps = 40`**, **`shift = 3.0`** (480p) or **`5.0`** (720p), **`text_guide_scale = 5.0`**, **`audio_guide_scale = 4.0`**.

In [ ]:
#@title 6. Run InfiniteTalk Generation & Save MP4 { display-mode: "form" }
PROMPT = "AUTO" #@param {type:"string"}
RESOLUTION = "infinitetalk-480" #@param ["infinitetalk-480", "infinitetalk-720"]
GENERATION_MODE = "streaming" #@param ["streaming", "clip"]
MAX_DURATION_SECONDS = 40.0 #@param {type:"number"}
#@markdown *(Set `MAX_DURATION_SECONDS = 0` to automatically generate the entire audio length without a cap.)*

FRAME_NUM = 81 #@param {type:"integer"}
MOTION_FRAMES = 9 #@param {type:"integer"}

#@markdown ### Sampling & Guidance Hyperparameters (`0` = Auto-Optimal for Loaded Preset)
SAMPLING_STEPS = 0 #@param {type:"integer"}
SAMPLE_SHIFT = 0.0 #@param {type:"number"}
TEXT_GUIDE_SCALE = 0.0 #@param {type:"number"}
AUDIO_GUIDE_SCALE = 0.0 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}

#@markdown ### Quality, Acceleration & V2V Options
COLOR_CORRECTION_STRENGTH = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.1}
USE_TEACACHE = False #@param {type:"boolean"}
TEACACHE_THRESH = 0.2 #@param {type:"slider", min:0.05, max:0.5, step:0.05}
USE_APG = False #@param {type:"boolean"}
APG_MOMENTUM = -0.75 #@param {type:"number"}
APG_NORM_THRESHOLD = 55.0 #@param {type:"number"}
SCENE_SEG = False #@param {type:"boolean"}
HIGH_QUALITY_SAVE = False #@param {type:"boolean"}
OFFLOAD_MODEL = True #@param {type:"boolean"}

import os
import gc
import time
import math
from datetime import datetime
from types import SimpleNamespace
import numpy as np
import soundfile as sf
import librosa
import pyloudnorm as pyln
import torch
from einops import rearrange
import IPython.display as ipd

from wan.utils.multitalk_utils import save_video_ffmpeg
from wan.utils.segvideo import shot_detect
from wan.utils.utils import split_wav_librosa, is_video

if "ACTIVE_INPUT" not in globals():
    raise RuntimeError("❌ Please run Step 5 first to configure your input image/video and audio.")
if "wan_i2v" not in globals() or wan_i2v is None:
    raise RuntimeError("❌ Please run Step 4 first to load InfiniteTalkPipeline.")

if (
    globals().get("SPEAKER_MODE") != globals().get("LOADED_SPEAKER_MODE")
    or globals().get("QUALITY_PRESET") != globals().get("LOADED_QUALITY_PRESET")
    or globals().get("QUANT_PRECISION") != globals().get("LOADED_QUANT_PRECISION")
):
    print("⚠️ Note: Step 3 dropdowns differ from the currently loaded pipeline in Step 4.")
    print(f"   Currently loaded: {LOADED_SPEAKER_MODE} / {LOADED_QUALITY_PRESET} / {LOADED_QUANT_PRECISION}. (Re-run Step 4 if you intended to switch checkpoints.)")

if (FRAME_NUM - 1) % 4 != 0:
    raise ValueError(f"❌ FRAME_NUM must be 4n + 1 (e.g. 81), got {FRAME_NUM}.")

# If running 720p on an L4 24GB GPU with 6B persistent params, automatically switch to 0 persistent params to prevent OOM
if RESOLUTION == "infinitetalk-720" and vram_total_gb < 35.0 and globals().get("LOADED_PERSISTENT_PARAMS") not in (0,):
    print("⚠️ 720p selected on a <35 GB GPU: Switching DiT to full layer-by-layer streaming (num_persistent_param_in_dit = 0)...")
    wan_i2v.vram_management = True
    wan_i2v.enable_vram_management(num_persistent_param_in_dit=0)
    LOADED_PERSISTENT_PARAMS = 0

is_lora_preset = (globals().get("LOADED_QUALITY_PRESET") == "fast_8step_lora")
steps = SAMPLING_STEPS if SAMPLING_STEPS > 0 else (8 if is_lora_preset else 40)
if SAMPLE_SHIFT > 0:
    shift = float(SAMPLE_SHIFT)
else:
    shift = 2.0 if is_lora_preset else (3.0 if RESOLUTION == "infinitetalk-480" else 5.0)
text_cfg = float(TEXT_GUIDE_SCALE) if TEXT_GUIDE_SCALE > 0 else (1.0 if is_lora_preset else 5.0)
audio_cfg = float(AUDIO_GUIDE_SCALE) if AUDIO_GUIDE_SCALE > 0 else (2.0 if is_lora_preset else 4.0)

prompt_text = ACTIVE_INPUT["default_prompt"] if PROMPT.strip().upper() == "AUTO" or not PROMPT.strip() else PROMPT.strip()

# Audio preparation helpers (with automatic short-audio padding & parallel track alignment)
def _loudness_norm(audio_array, sr=16000, lufs=-23):
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio_array)
    if abs(loudness) > 100:
        return audio_array
    return pyln.normalize.loudness(audio_array, loudness, lufs)

def _load_single_speech(path_str, sr=16000):
    ext = os.path.splitext(path_str)[1].lower()
    if ext in [".mp4", ".mov", ".avi", ".mkv", ".webm"]:
        tmp_wav = os.path.join(AUDIO_TEMP_DIR, "_extracted_input_audio.wav")
        subprocess.run(
            ["ffmpeg", "-y", "-i", path_str, "-vn", "-acodec", "pcm_s16le", "-ar", str(sr), "-ac", "1", tmp_wav],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
        )
        arr, _ = librosa.load(tmp_wav, sr=sr)
        os.remove(tmp_wav)
    else:
        arr, _ = librosa.load(path_str, sr=sr)
    return _loudness_norm(arr, sr)

def _prepare_multi_speech(p1, p2, mode_str, sr=16000):
    s1 = _load_single_speech(p1, sr)
    s2 = _load_single_speech(p2, sr)
    if mode_str == "para":
        max_len = max(len(s1), len(s2))
        if len(s1) < max_len:
            s1 = np.pad(s1, (0, max_len - len(s1)))
        if len(s2) < max_len:
            s2 = np.pad(s2, (0, max_len - len(s2)))
        ns1, ns2 = s1, s2
    else:  # 'add' sequential turn-taking
        ns1 = np.concatenate([s1, np.zeros(len(s2), dtype=s1.dtype)])
        ns2 = np.concatenate([np.zeros(len(s1), dtype=s2.dtype), s2])
    return ns1, ns2, ns1 + ns2

def _extract_wav2vec_emb(speech_array, min_frames, sr=16000):
    orig_frames = max(1, int(round((len(speech_array) / sr) * 25)))
    # Ensure audio is long enough to satisfy InfiniteTalk's chunk window (> FRAME_NUM)
    target_min_samples = int(math.ceil(((min_frames + 4) / 25.0) * sr))
    if len(speech_array) < target_min_samples:
        padded_speech = np.pad(speech_array, (0, target_min_samples - len(speech_array)))
    else:
        padded_speech = speech_array
    seq_len = int((len(padded_speech) / sr) * 25)
    audio_feat = np.squeeze(wav2vec_feature_extractor(padded_speech, sampling_rate=sr).input_values)
    audio_feat = torch.from_numpy(audio_feat).float().unsqueeze(0).to(wan_i2v.device)
    with torch.no_grad():
        audio_encoder.to(wan_i2v.device)
        embeddings = audio_encoder(audio_feat, seq_len=seq_len, output_hidden_states=True)
        audio_encoder.cpu()
    torch.cuda.empty_cache()
    audio_emb = torch.stack(embeddings.hidden_states[1:], dim=1).squeeze(0)
    audio_emb = rearrange(audio_emb, "b s d -> s b d").cpu().detach()
    return audio_emb, orig_frames

# Prepare segments (supports scene_seg for V2V)
run_audio_dir = os.path.join(AUDIO_TEMP_DIR, datetime.now().strftime("run_%H%M%S"))
os.makedirs(run_audio_dir, exist_ok=True)

cond_video = ACTIVE_INPUT["cond_video"]
audio1 = ACTIVE_INPUT["audio1"]
audio2 = ACTIVE_INPUT["audio2"]
is_multi = audio2 is not None

conds_list = []
if SCENE_SEG and is_video(cond_video):
    time_list, cond_list = shot_detect(cond_video, run_audio_dir)
    if len(time_list) == 0:
        conds_list = [[cond_video], [audio1]] + ([[audio2]] if is_multi else [])
    else:
        a1_list = split_wav_librosa(audio1, time_list, run_audio_dir)
        conds_list = [cond_list, a1_list]
        if is_multi:
            conds_list.append(split_wav_librosa(audio2, time_list, run_audio_dir))
else:
    conds_list = [[cond_video], [audio1]] + ([[audio2]] if is_multi else [])

# Prepare master audio track for final muxing
if is_multi:
    _, _, full_sum_speech = _prepare_multi_speech(audio1, audio2, ACTIVE_INPUT["audio_type"])
else:
    full_sum_speech = _load_single_speech(audio1)

master_wav_path = os.path.join(run_audio_dir, "master_audio.wav")
sf.write(master_wav_path, full_sum_speech, 16000)

total_orig_frames = max(1, int(round((len(full_sum_speech) / 16000.0) * 25)))
if GENERATION_MODE == "clip":
    max_frames_num = FRAME_NUM
elif MAX_DURATION_SECONDS and MAX_DURATION_SECONDS > 0:
    max_frames_num = min(int(round(MAX_DURATION_SECONDS * 25)), max(total_orig_frames, FRAME_NUM))
else:
    max_frames_num = max(total_orig_frames, FRAME_NUM)

extra_args = SimpleNamespace(
    use_teacache=bool(USE_TEACACHE),
    teacache_thresh=float(TEACACHE_THRESH),
    size=RESOLUTION,
    use_apg=bool(USE_APG),
    apg_momentum=float(APG_MOMENTUM),
    apg_norm_threshold=float(APG_NORM_THRESHOLD),
)

print("=" * 72)
print(f"🚀 Starting InfiniteTalk Generation")
print(f"   • Preset / Quant   : {LOADED_QUALITY_PRESET} ({LOADED_QUANT_PRECISION.upper()}, {LOADED_SPEAKER_MODE})")
print(f"   • Resolution       : {RESOLUTION} | Mode: {GENERATION_MODE} (chunk={FRAME_NUM}, motion={MOTION_FRAMES})")
print(f"   • Target Frames    : {min(total_orig_frames, max_frames_num)} frames ({min(total_orig_frames, max_frames_num)/25.0:.2f}s @ 25 FPS)")
print(f"   • Hyperparameters  : steps={steps}, shift={shift}, text_cfg={text_cfg}, audio_cfg={audio_cfg}, seed={SEED}")
print(f"   • Color Correction : {COLOR_CORRECTION_STRENGTH} | TeaCache: {USE_TEACACHE} | APG: {USE_APG}")
print("=" * 72)

t_start = time.time()
generated_segments = []

for seg_idx, items in enumerate(zip(*conds_list)):
    seg_cond_video = items[0]
    input_clip = {
        "prompt": prompt_text,
        "cond_video": seg_cond_video,
    }
    if is_multi:
        input_clip["audio_type"] = ACTIVE_INPUT["audio_type"]
        if ACTIVE_INPUT["bbox"]:
            input_clip["bbox"] = ACTIVE_INPUT["bbox"]
        s1_seg, s2_seg, sum_seg = _prepare_multi_speech(items[1], items[2], ACTIVE_INPUT["audio_type"])
        emb1, orig_f1 = _extract_wav2vec_emb(s1_seg, FRAME_NUM)
        emb2, orig_f2 = _extract_wav2vec_emb(s2_seg, FRAME_NUM)
        seg_orig_frames = max(orig_f1, orig_f2)
        emb1_p = os.path.join(run_audio_dir, f"seg{seg_idx}_1.pt")
        emb2_p = os.path.join(run_audio_dir, f"seg{seg_idx}_2.pt")
        torch.save(emb1, emb1_p)
        torch.save(emb2, emb2_p)
        input_clip["cond_audio"] = {"person1": emb1_p, "person2": emb2_p}
    else:
        s1_seg = _load_single_speech(items[1])
        emb1, seg_orig_frames = _extract_wav2vec_emb(s1_seg, FRAME_NUM)
        emb1_p = os.path.join(run_audio_dir, f"seg{seg_idx}_1.pt")
        torch.save(emb1, emb1_p)
        input_clip["cond_audio"] = {"person1": emb1_p}

    seg_max_frames = FRAME_NUM if GENERATION_MODE == "clip" else max(seg_orig_frames, FRAME_NUM)
    if MAX_DURATION_SECONDS and MAX_DURATION_SECONDS > 0 and GENERATION_MODE != "clip":
        seg_max_frames = min(seg_max_frames, max_frames_num)

    video_tensor = wan_i2v.generate_infinitetalk(
        input_clip,
        size_buckget=RESOLUTION,
        motion_frame=MOTION_FRAMES,
        frame_num=FRAME_NUM,
        shift=shift,
        sampling_steps=steps,
        text_guide_scale=text_cfg,
        audio_guide_scale=audio_cfg,
        seed=SEED,
        offload_model=OFFLOAD_MODEL,
        max_frames_num=seg_max_frames,
        color_correction_strength=float(COLOR_CORRECTION_STRENGTH),
        extra_args=extra_args,
    )

    # Trim any silence padding added for short clips (< FRAME_NUM)
    target_trim = min(video_tensor.shape[1], seg_orig_frames, max_frames_num)
    video_tensor = video_tensor[:, :target_trim]
    generated_segments.append(video_tensor)

sum_video = torch.cat(generated_segments, dim=1)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_tag = os.path.splitext(os.path.basename(cond_video))[0][:24]
out_prefix = os.path.join(OUTPUT_DIR, f"InfiniteTalk_{LOADED_SPEAKER_MODE}_{RESOLUTION.split('-')[1]}p_{steps}steps_{safe_tag}_{timestamp}")

save_video_ffmpeg(sum_video, out_prefix, [master_wav_path], fps=25, high_quality_save=HIGH_QUALITY_SAVE)
final_mp4_path = out_prefix + ".mp4"
elapsed = time.time() - t_start

print("=" * 72)
print(f"🎉 Generation Complete in {elapsed/60:.1f} min ({elapsed:.1f}s)!")
print(f"💾 Saved MP4 ({sum_video.shape[1]} frames / {sum_video.shape[1]/25.0:.2f}s): {final_mp4_path}")
print("=" * 72)

if os.path.exists(final_mp4_path):
    try:
        from IPython.display import FileLink
        display(FileLink(final_mp4_path))
    except Exception:
        pass
    if os.path.getsize(final_mp4_path) < 40 * (1024 ** 2):
        display(ipd.Video(final_mp4_path, embed=True, width=640))
    else:
        print("ℹ️ Video is >40 MB; download or play directly from your Google Drive outputs folder:", final_mp4_path)

# Expose reusable helper for Gradio UI (Step 8) and Batch Queue (Step 9)
def run_infinitetalk_job(
    cond_video_p,
    audio1_p,
    audio2_p=None,
    audio_type_str="add",
    bbox_dict_in=None,
    prompt_in="A person is talking naturally with realistic facial expressions.",
    resolution_in="infinitetalk-480",
    gen_mode_in="streaming",
    max_dur_sec=40.0,
    frame_num_in=81,
    motion_frames_in=9,
    sampling_steps_in=0,
    sample_shift_in=0.0,
    text_cfg_in=0.0,
    audio_cfg_in=0.0,
    seed_in=42,
    color_corr_in=0.5,
    use_teacache_in=False,
    teacache_thresh_in=0.2,
    scene_seg_in=False,
    high_quality_in=False,
):
    global LOADED_PERSISTENT_PARAMS
    if resolution_in == "infinitetalk-720" and vram_total_gb < 35.0 and globals().get("LOADED_PERSISTENT_PARAMS") not in (0,):
        wan_i2v.vram_management = True
        wan_i2v.enable_vram_management(num_persistent_param_in_dit=0)
        LOADED_PERSISTENT_PARAMS = 0

    lora_active = (globals().get("LOADED_QUALITY_PRESET") == "fast_8step_lora")
    st = int(sampling_steps_in) if int(sampling_steps_in) > 0 else (8 if lora_active else 40)
    sh = float(sample_shift_in) if float(sample_shift_in) > 0 else (2.0 if lora_active else (3.0 if resolution_in == "infinitetalk-480" else 5.0))
    t_cfg = float(text_cfg_in) if float(text_cfg_in) > 0 else (1.0 if lora_active else 5.0)
    a_cfg = float(audio_cfg_in) if float(audio_cfg_in) > 0 else (2.0 if lora_active else 4.0)

    job_dir = os.path.join(AUDIO_TEMP_DIR, datetime.now().strftime("job_%H%M%S_%f"))
    os.makedirs(job_dir, exist_ok=True)
    multi_flag = bool(audio2_p)

    if multi_flag and globals().get("LOADED_SPEAKER_MODE") != "multi":
        raise RuntimeError("❌ Dual-speaker generation requires SPEAKER_MODE = 'multi' in Steps 3 & 4.")

    if scene_seg_in and is_video(cond_video_p):
        t_list, c_list = shot_detect(cond_video_p, job_dir)
        if len(t_list) == 0:
            c_items = [[cond_video_p], [audio1_p]] + ([[audio2_p]] if multi_flag else [])
        else:
            a1_l = split_wav_librosa(audio1_p, t_list, job_dir)
            c_items = [c_list, a1_l] + ([split_wav_librosa(audio2_p, t_list, job_dir)] if multi_flag else [])
    else:
        c_items = [[cond_video_p], [audio1_p]] + ([[audio2_p]] if multi_flag else [])

    if multi_flag:
        _, _, master_sp = _prepare_multi_speech(audio1_p, audio2_p, audio_type_str)
    else:
        master_sp = _load_single_speech(audio1_p)

    m_wav = os.path.join(job_dir, "master_audio.wav")
    sf.write(m_wav, master_sp, 16000)

    tot_frames = max(1, int(round((len(master_sp) / 16000.0) * 25)))
    if gen_mode_in == "clip":
        max_f = int(frame_num_in)
    elif max_dur_sec and float(max_dur_sec) > 0:
        max_f = min(int(round(float(max_dur_sec) * 25)), max(tot_frames, int(frame_num_in)))
    else:
        max_f = max(tot_frames, int(frame_num_in))

    ex_args = SimpleNamespace(
        use_teacache=bool(use_teacache_in),
        teacache_thresh=float(teacache_thresh_in),
        size=resolution_in,
        use_apg=False,
        apg_momentum=-0.75,
        apg_norm_threshold=55.0,
    )

    segs = []
    for s_i, it in enumerate(zip(*c_items)):
        ic = {"prompt": prompt_in, "cond_video": it[0]}
        if multi_flag:
            ic["audio_type"] = audio_type_str
            if bbox_dict_in:
                ic["bbox"] = bbox_dict_in
            s1_s, s2_s, _ = _prepare_multi_speech(it[1], it[2], audio_type_str)
            e1, of1 = _extract_wav2vec_emb(s1_s, int(frame_num_in))
            e2, of2 = _extract_wav2vec_emb(s2_s, int(frame_num_in))
            s_orig = max(of1, of2)
            p1_pt = os.path.join(job_dir, f"s{s_i}_1.pt")
            p2_pt = os.path.join(job_dir, f"s{s_i}_2.pt")
            torch.save(e1, p1_pt)
            torch.save(e2, p2_pt)
            ic["cond_audio"] = {"person1": p1_pt, "person2": p2_pt}
        else:
            s1_s = _load_single_speech(it[1])
            e1, s_orig = _extract_wav2vec_emb(s1_s, int(frame_num_in))
            p1_pt = os.path.join(job_dir, f"s{s_i}_1.pt")
            torch.save(e1, p1_pt)
            ic["cond_audio"] = {"person1": p1_pt}

        s_max = int(frame_num_in) if gen_mode_in == "clip" else max(s_orig, int(frame_num_in))
        if max_dur_sec and float(max_dur_sec) > 0 and gen_mode_in != "clip":
            s_max = min(s_max, max_f)

        vt = wan_i2v.generate_infinitetalk(
            ic,
            size_buckget=resolution_in,
            motion_frame=int(motion_frames_in),
            frame_num=int(frame_num_in),
            shift=sh,
            sampling_steps=st,
            text_guide_scale=t_cfg,
            audio_guide_scale=a_cfg,
            seed=int(seed_in),
            offload_model=True,
            max_frames_num=s_max,
            color_correction_strength=float(color_corr_in),
            extra_args=ex_args,
        )
        t_trim = min(vt.shape[1], s_orig, max_f)
        segs.append(vt[:, :t_trim])

    merged_v = torch.cat(segs, dim=1)
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    tag_str = os.path.splitext(os.path.basename(cond_video_p))[0][:24]
    out_pref = os.path.join(OUTPUT_DIR, f"InfiniteTalk_{LOADED_SPEAKER_MODE}_{resolution_in.split('-')[1]}p_{st}steps_{tag_str}_{ts_str}")
    save_video_ffmpeg(merged_v, out_pref, [m_wav], fps=25, high_quality_save=high_quality_in)
    return out_pref + ".mp4"

---
## Step 7 — Long-Video (`>60s`) Pan/Zoom Image-to-Video Helper *(Optional)*
As noted in the official [`MeiGen-AI/InfiniteTalk` README](https://github.com/MeiGen-AI/InfiniteTalk#-quick-inference):
> *"I2V generation generates good results from a single image for up to 1 minute. Beyond 1 minute, color shifts become more pronounced. One trick for high-quality generation beyond 1 min is to convert the static image into a video by translating or zooming in the image."*

Run this optional utility cell to convert your static reference image from **Step 5** into a smooth, subtle pan/zoom `.mp4` conditioning video matching your audio duration, and automatically set it as `ACTIVE_INPUT["cond_video"]` for **Step 6**!

In [ ]:
#@title 7. Convert Static Reference Image into Subtle Pan/Zoom Conditioning Video { display-mode: "form" }
ZOOM_SCALE = 1.08 #@param {type:"slider", min:1.0, max:1.25, step:0.01}
PAN_X_PERCENT = 2.0 #@param {type:"slider", min:-10.0, max:10.0, step:0.5}
PAN_Y_PERCENT = -1.5 #@param {type:"slider", min:-10.0, max:10.0, step:0.5}
SET_AS_ACTIVE_INPUT = True #@param {type:"boolean"}

import os
import subprocess
import cv2
import numpy as np

if "ACTIVE_INPUT" not in globals():
    raise RuntimeError("❌ Please run Step 5 first.")

src_img_path = ACTIVE_INPUT["cond_video"]
if ACTIVE_INPUT["is_video"]:
    print(f"ℹ️ Current ACTIVE_INPUT ({src_img_path}) is already a video. No conversion needed.")
else:
    cv_img = cv2.imread(src_img_path)
    if cv_img is None:
        raise ValueError(f"❌ Could not read image: {src_img_path}")
    h, w = cv_img.shape[:2]
    fps = 25
    total_frames = max(85, ACTIVE_INPUT["est_frames"] + 10)
    raw_mp4 = os.path.join(INPUT_DIR, "_panzoom_raw.mp4")
    out_mp4 = os.path.join(INPUT_DIR, f"{os.path.splitext(os.path.basename(src_img_path))[0]}_panzoom.mp4")

    writer = cv2.VideoWriter(raw_mp4, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    print(f"🎞️ Generating {total_frames} smooth pan/zoom frames ({total_frames/fps:.1f}s @ 25 FPS)...")
    for idx in range(total_frames):
        # Smooth ping-pong cosine easing across the video duration
        alpha = 0.5 * (1.0 - np.cos(2.0 * np.pi * (idx / max(1, total_frames - 1))))
        cur_scale = 1.0 + (ZOOM_SCALE - 1.0) * alpha
        tx = (w * (PAN_X_PERCENT / 100.0)) * alpha
        ty = (h * (PAN_Y_PERCENT / 100.0)) * alpha
        M = np.float32([
            [cur_scale, 0, (1.0 - cur_scale) * (w / 2.0) + tx],
            [0, cur_scale, (1.0 - cur_scale) * (h / 2.0) + ty],
        ])
        frame = cv2.warpAffine(cv_img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        writer.write(frame)
    writer.release()

    subprocess.run(
        ["ffmpeg", "-y", "-i", raw_mp4, "-c:v", "libx264", "-preset", "fast", "-crf", "18", out_mp4],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
    )
    if os.path.exists(raw_mp4):
        os.remove(raw_mp4)

    print(f"✅ Created smooth pan/zoom conditioning video: {out_mp4}")
    if SET_AS_ACTIVE_INPUT:
        ACTIVE_INPUT["cond_video"] = out_mp4
        ACTIVE_INPUT["is_video"] = True
        print("👉 Updated ACTIVE_INPUT['cond_video']! You can now re-run Step 6 to generate in V2V streaming mode.")

---
## Step 8 — Launch Interactive Gradio Studio UI *(Optional)*
Prefer a browser studio over Colab form cells? Run this cell to launch a 2-tab **Gradio Studio UI** (with public `gradio.live` share link) supporting:
1. **🖼️ Single-Person I2V & Sparse-Frame V2V Video Dubbing**
2. **👥 Multi-Speaker Conversation (`Multi-I2V` / `Multi-V2V`)**
3. **📂 Cross-Notebook Drive Audio Picker**: Load recent `.wav`/`.mp3` files generated by `Qwen3-TTS`, `Breeze-TTS-2`, `VoxCPM2`, `Audio_PostProcessor`, or `MiniMax-Music3` directly from your Google Drive with one click!

In [ ]:
#@title 8. Launch InfiniteTalk Gradio Studio UI { display-mode: "form" }
import os
import gradio as gr
from IPython.display import clear_output

if "wan_i2v" not in globals() or wan_i2v is None:
    raise RuntimeError("❌ Please run Steps 1–4 first to initialize InfiniteTalkPipeline.")
if "run_infinitetalk_job" not in globals():
    raise RuntimeError("❌ Please run Step 5 & Step 6 once (or define run_infinitetalk_job) before launching the UI.")

def _refresh_drive_choices():
    items = discover_aei_drive_audio(max_files=50)
    return gr.update(choices=items, value=items[0] if items else None)

def _ui_run_single(cond_file, aud_file, prompt_str, res_str, mode_str, max_dur, scene_seg_flag, color_corr, teacache_flag, seed_val):
    if not cond_file or not aud_file:
        raise gr.Error("Please provide both a Reference Image/Video and Driving Audio.")
    out_mp4 = run_infinitetalk_job(
        cond_video_p=cond_file,
        audio1_p=aud_file,
        prompt_in=prompt_str,
        resolution_in=res_str,
        gen_mode_in=mode_str,
        max_dur_sec=float(max_dur),
        seed_in=int(seed_val),
        color_corr_in=float(color_corr),
        use_teacache_in=bool(teacache_flag),
        scene_seg_in=bool(scene_seg_flag),
    )
    return out_mp4, f"✅ Saved to Google Drive: {out_mp4}"

def _ui_run_multi(cond_file, aud1_file, aud2_file, atype_label, bbox1_s, bbox2_s, prompt_str, res_str, mode_str, max_dur, seed_val):
    if not cond_file or not aud1_file or not aud2_file:
        raise gr.Error("Please provide a Reference Image/Video and both Speaker 1 & Speaker 2 Audio tracks.")
    atype = "add" if "Sequential" in atype_label else "para"
    b_dict = None
    if bbox1_s.strip() and bbox2_s.strip():
        p1 = [int(float(v.strip())) for v in bbox1_s.split(",") if v.strip()]
        p2 = [int(float(v.strip())) for v in bbox2_s.split(",") if v.strip()]
        if len(p1) == 4 and len(p2) == 4:
            b_dict = {"person1": p1, "person2": p2}
    out_mp4 = run_infinitetalk_job(
        cond_video_p=cond_file,
        audio1_p=aud1_file,
        audio2_p=aud2_file,
        audio_type_str=atype,
        bbox_dict_in=b_dict,
        prompt_in=prompt_str,
        resolution_in=res_str,
        gen_mode_in=mode_str,
        max_dur_sec=float(max_dur),
        seed_in=int(seed_val),
    )
    return out_mp4, f"✅ Saved to Google Drive: {out_mp4}"

initial_drive_audios = discover_aei_drive_audio(max_files=50)

with gr.Blocks(title="InfiniteTalk Studio (Colab)") as demo:
    hdr_md = gr.Markdown(
        f"## 🎬 InfiniteTalk Studio — Loaded: `{LOADED_SPEAKER_MODE}` / `{LOADED_QUALITY_PRESET}` (`{LOADED_QUANT_PRECISION.upper()}`)\n"
        f"Outputs are saved automatically to `{OUTPUT_DIR}`."
    )
    demo.load(
        fn=lambda: f"## 🎬 InfiniteTalk Studio — Loaded: `{LOADED_SPEAKER_MODE}` / `{LOADED_QUALITY_PRESET}` (`{LOADED_QUANT_PRECISION.upper()}`)\nOutputs are saved automatically to `{OUTPUT_DIR}`.",
        outputs=[hdr_md],
    )

    with gr.Accordion("📂 Cross-Notebook Drive Audio Picker (Load Recent Audio from TTS / Music / Post-Processor)", open=False):
        with gr.Row():
            drive_aud_dd = gr.Dropdown(
                choices=initial_drive_audios,
                value=initial_drive_audios[0] if initial_drive_audios else None,
                label="Recent Audio Files on Google Drive",
                info="Scans AEI_TTS_Out, AEI_3D_Out, AEI_TTS_Cache/voices, and InfiniteTalk_Colab/inputs",
                scale=4,
            )
            refresh_drive_btn = gr.Button("🔄 Refresh Drive Audio", scale=1)

    with gr.Tabs():
        with gr.Tab("1. Single-Person I2V & Sparse-Frame V2V Dubbing"):
            with gr.Row():
                with gr.Column(scale=1):
                    s_cond = gr.File(
                        label="Reference Portrait Image (.png/.jpg) or Video (.mp4 for V2V Dubbing)",
                        type="filepath",
                    )
                    s_aud = gr.Audio(
                        label="Driving Speech / Singing Audio",
                        type="filepath",
                    )
                    load_s_aud_btn = gr.Button("⬅️ Load Selected Drive Audio into Single-Person Audio", size="sm")
                    s_prompt = gr.Textbox(
                        label="Scene & Motion Prompt",
                        lines=3,
                        value="A person is talking naturally with expressive facial movements and realistic head gestures.",
                        info="Describe the character, lighting, and expression.",
                    )
                    with gr.Row():
                        s_res = gr.Dropdown(
                            ["infinitetalk-480", "infinitetalk-720"],
                            value="infinitetalk-480",
                            label="Resolution",
                            info="480p recommended on L4 24GB; 720p supported on A100 40GB+",
                        )
                        s_mode = gr.Dropdown(
                            ["streaming", "clip"],
                            value="streaming",
                            label="Generation Mode",
                            info="'streaming' generates full audio length; 'clip' generates 1 chunk (3.24s)",
                        )
                    with gr.Row():
                        s_dur = gr.Number(
                            value=40.0,
                            label="Max Duration Seconds (0 = Full Audio)",
                            info="Caps streaming duration in seconds (set 0 for unlimited)",
                        )
                        s_seed = gr.Number(
                            value=42,
                            precision=0,
                            label="Random Seed",
                            info="Fixed integer seed for reproducible motion",
                        )
                    with gr.Accordion("V2V Dubbing & Acceleration Options", open=False):
                        s_scene = gr.Checkbox(
                            value=False,
                            label="Auto Shot-Boundary Split (PySceneDetect for multi-shot V2V videos)",
                            info="Splits input video at camera cuts before dubbing each shot",
                        )
                        s_cc = gr.Slider(
                            0.0, 1.0, value=0.5, step=0.1,
                            label="Lab Color Correction Strength",
                            info="Prevents color drift across long streaming generations",
                        )
                        s_tc = gr.Checkbox(
                            value=False,
                            label="Enable TeaCache Acceleration",
                            info="Skips redundant DiT blocks when consecutive latents change slowly",
                        )
                    s_btn = gr.Button("🚀 Generate InfiniteTalk Video", variant="primary")
                with gr.Column(scale=1):
                    s_vid = gr.Video(label="Generated / Dubbed Video (25 FPS)")
                    s_status = gr.Textbox(label="Output Status", interactive=False)

            s_btn.click(
                fn=_ui_run_single,
                inputs=[s_cond, s_aud, s_prompt, s_res, s_mode, s_dur, s_scene, s_cc, s_tc, s_seed],
                outputs=[s_vid, s_status],
            )
            load_s_aud_btn.click(fn=lambda p: p, inputs=[drive_aud_dd], outputs=[s_aud])

        with gr.Tab("2. Multi-Speaker Conversation (Multi-I2V / Multi-V2V)"):
            with gr.Row():
                with gr.Column(scale=1):
                    m_cond = gr.File(
                        label="Two-Person Reference Image (.png/.jpg) or Video (.mp4)",
                        type="filepath",
                    )
                    with gr.Row():
                        m_aud1 = gr.Audio(label="Speaker 1 (Left) Audio", type="filepath")
                        m_aud2 = gr.Audio(label="Speaker 2 (Right) Audio", type="filepath")
                    with gr.Row():
                        load_m1_btn = gr.Button("⬅️ Load Drive Audio -> Speaker 1", size="sm")
                        load_m2_btn = gr.Button("⬅️ Load Drive Audio -> Speaker 2", size="sm")
                    m_atype = gr.Radio(
                        ["Sequential Turn-Taking (add)", "Parallel / Simultaneous (para)"],
                        value="Sequential Turn-Taking (add)",
                        label="Multi-Audio Mode",
                        info="'add' plays Speaker 1 then Speaker 2; 'para' plays both simultaneously",
                    )
                    m_prompt = gr.Textbox(
                        label="Multi-Speaker Scene Prompt",
                        lines=3,
                        value="In a casual setting, two people are engaged in a lively conversation with natural facial expressions.",
                        info="Describe both speakers and the environment",
                    )
                    with gr.Row():
                        m_bbox1 = gr.Textbox(label="Speaker 1 BBox (x1, y1, x2, y2)", placeholder="Auto left half", info="Optional row/col pixel box")
                        m_bbox2 = gr.Textbox(label="Speaker 2 BBox (x1, y1, x2, y2)", placeholder="Auto right half", info="Optional row/col pixel box")
                    with gr.Row():
                        m_res = gr.Dropdown(["infinitetalk-480", "infinitetalk-720"], value="infinitetalk-480", label="Resolution")
                        m_mode = gr.Dropdown(["streaming", "clip"], value="streaming", label="Mode")
                        m_dur = gr.Number(value=40.0, label="Max Seconds (0 = Full)")
                        m_seed = gr.Number(value=42, precision=0, label="Seed")
                    m_btn = gr.Button("🚀 Generate Multi-Speaker Video", variant="primary")
                with gr.Column(scale=1):
                    m_vid = gr.Video(label="Generated Multi-Speaker Video (25 FPS)")
                    m_status = gr.Textbox(label="Output Status", interactive=False)

            m_btn.click(
                fn=_ui_run_multi,
                inputs=[m_cond, m_aud1, m_aud2, m_atype, m_bbox1, m_bbox2, m_prompt, m_res, m_mode, m_dur, m_seed],
                outputs=[m_vid, m_status],
            )
            load_m1_btn.click(fn=lambda p: p, inputs=[drive_aud_dd], outputs=[m_aud1])
            load_m2_btn.click(fn=lambda p: p, inputs=[drive_aud_dd], outputs=[m_aud2])

    refresh_drive_btn.click(fn=_refresh_drive_choices, inputs=[], outputs=[drive_aud_dd])

clear_output(wait=True)
demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, show_error=True)

---
## Step 9 — Batch Scene Processor (JSON Queue + Resume Log) *(Optional)*
Queue multiple single-person I2V, V2V video dubbing, or multi-speaker scenes from a JSON file on Google Drive (`batch_scenes.json`).
- **Auto-Resume**: Completed scenes are recorded by content SHA-256 hash in `batch_infinitetalk.log.jsonl` so interrupted Colab sessions pick up right where they left off.
- **Fault-Tolerant**: Each scene runs inside a `try/except` block so one bad path never aborts the rest of your queue.

In [ ]:
#@title 9. Run Batch Scene Queue from JSON (with Auto-Resume) { display-mode: "form" }
BATCH_JSON_PATH = "" #@param {type:"string"}
#@markdown *(Leave blank to use `<WORKSPACE_DIR>/batch_scenes.json`. If the file does not exist yet, a 2-scene template is created automatically.)*
RESET_RESUME_LOG = False #@param {type:"boolean"}

import os
import json
import hashlib
import traceback

if "run_infinitetalk_job" not in globals():
    raise RuntimeError("❌ Please run Steps 1–6 first so the pipeline and run_infinitetalk_job() are initialized.")

batch_json_file = BATCH_JSON_PATH.strip() or os.path.join(WORKSPACE_DIR, "batch_scenes.json")
resume_log_file = os.path.join(WORKSPACE_DIR, "batch_infinitetalk.log.jsonl")

if RESET_RESUME_LOG and os.path.exists(resume_log_file):
    os.remove(resume_log_file)
    print("🗑️ Cleared batch resume log:", resume_log_file)

if not os.path.exists(batch_json_file):
    sample_scenes = [
        {
            "name": "scene_01_singing_i2v",
            "cond_video": os.path.join(LOCAL_REPO_DIR, "examples", "single", "ref_image.png"),
            "audio1": os.path.join(LOCAL_REPO_DIR, "examples", "single", "1.wav"),
            "prompt": DEFAULT_DEMO_PROMPTS["demo_single_singing_image"],
            "resolution": "infinitetalk-480",
            "mode": "clip",
            "max_duration_sec": 10.0,
            "seed": 42,
        },
        {
            "name": "scene_02_dubbing_v2v",
            "cond_video": os.path.join(LOCAL_REPO_DIR, "examples", "single", "ref_video.mp4"),
            "audio1": os.path.join(LOCAL_REPO_DIR, "examples", "single", "1.wav"),
            "prompt": DEFAULT_DEMO_PROMPTS["demo_single_talking_video"],
            "resolution": "infinitetalk-480",
            "mode": "clip",
            "max_duration_sec": 10.0,
            "seed": 43,
        },
    ]
    with open(batch_json_file, "w", encoding="utf-8") as f:
        json.dump(sample_scenes, f, indent=2, ensure_ascii=False)
    print(f"📝 Created sample 2-scene batch template at: {batch_json_file}")

with open(batch_json_file, "r", encoding="utf-8") as f:
    scenes_queue = json.load(f)

completed_hashes = {}
if os.path.exists(resume_log_file):
    with open(resume_log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if rec.get("output_mp4") and os.path.exists(rec["output_mp4"]):
                    completed_hashes[rec["hash"]] = rec["output_mp4"]
            except Exception:
                pass

print(f"📋 Loaded {len(scenes_queue)} scene(s) from {batch_json_file} ({len(completed_hashes)} already completed).")

for idx, scene in enumerate(scenes_queue, 1):
    scene_key = hashlib.sha256(json.dumps(scene, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    scene_name = scene.get("name", f"scene_{idx:03d}")
    if scene_key in completed_hashes:
        print(f"⏭️ [{idx}/{len(scenes_queue)}] SKIP {scene_name} (already on disk: {completed_hashes[scene_key]})")
        continue

    print(f"\n🎬 [{idx}/{len(scenes_queue)}] Running {scene_name} (hash={scene_key})...")
    try:
        out_mp4 = run_infinitetalk_job(
            cond_video_p=scene["cond_video"],
            audio1_p=scene["audio1"],
            audio2_p=scene.get("audio2"),
            audio_type_str=scene.get("audio_type", "add"),
            bbox_dict_in=scene.get("bbox"),
            prompt_in=scene.get("prompt", "A person is talking naturally with expressive facial movements."),
            resolution_in=scene.get("resolution", "infinitetalk-480"),
            gen_mode_in=scene.get("mode", "streaming"),
            max_dur_sec=float(scene.get("max_duration_sec", 40.0)),
            seed_in=int(scene.get("seed", 42)),
            color_corr_in=float(scene.get("color_correction_strength", 0.5)),
            use_teacache_in=bool(scene.get("use_teacache", False)),
            scene_seg_in=bool(scene.get("scene_seg", False)),
        )
        with open(resume_log_file, "a", encoding="utf-8") as lf:
            lf.write(json.dumps({"hash": scene_key, "name": scene_name, "output_mp4": out_mp4}) + "\n")
        print(f"✅ [{idx}/{len(scenes_queue)}] Completed {scene_name} -> {out_mp4}")
    except Exception as exc:
        print(f"❌ [{idx}/{len(scenes_queue)}] FAILED {scene_name}: {exc}")
        traceback.print_exc()